<a href="https://colab.research.google.com/github/AdiRatnam/Hands_on_Pytorch/blob/main/getting_started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autograd

package which provide automatic differentiation for all operation on Tensors. since we need to compute gradient, so it compute it applying chain rule


In [2]:
import torch

In [3]:
x = torch.randn(3, requires_grad=True)
#requires_grad = True - tracks all operation happening on the tensors

y = x + 2
print(x)
print(y)
print(x.grad_fn)
print(y.grad_fn)

tensor([-0.4322,  0.3722, -1.1639], requires_grad=True)
tensor([1.5678, 2.3722, 0.8361], grad_fn=<AddBackward0>)
None


In [5]:
# we compute more function :-
z = y * y * 3
print(z)
z = z.mean()
print(z)

tensor([ 7.3735, 16.8823,  2.0973], grad_fn=<MulBackward0>)
tensor(8.7844, grad_fn=<MeanBackward0>)


In [7]:
# Let's compute the gradients with backpropagation
# When we finish our computation we can call .backward() and have all the gradients computed automatically.
# The gradient for this tensor will be accumulated into .grad attribute.
# It is the partial derivate of the function w.r.t. the tensor

print(x.grad) # it will show none, we will get the gradient after performing backprop
z.backward()
print(x.grad) # dz/dx

None
tensor([3.1355, 4.7444, 1.6722])


In [8]:
# !!! Careful!!! backward() accumulates the gradient for this tensor into .grad attribute.
# we need to make sure that we will empty them in each iteration
# !!! We need to be careful during optimization !!! optimizer.zero_grad()

# Stop a tensor from tracking history:
For example during the training loop when we want to update our weights, or after training during evaluation. These operations should not be part of the gradient computation. To prevent this, we can use:

- `x.requires_grad_(False)`

- `x.detach()`

- wrap in `with torch.no_grad()`

In [11]:
# 1 - .requires_grad_( ) changes an existing flag in-place.
a = torch.randn(2, 2) # - here it set as false as we pass nothing
b = (a * a).sum()
print(a.requires_grad)  # shows false
print(b.grad_fn)  # shows none

a.requires_grad_(True) # here we set
b = (a * a).sum()
print(a.requires_grad)
print(b.grad_fn)

False
None
True


In [12]:
# 2 - .detach(): create a new Tensor with the same content but no gradient computation:
a = torch.randn(2, 2, requires_grad=True)
b = a.detach()
print(a.requires_grad)
print(b.requires_grad)

True
False


In [13]:
# 3 - wrap in with torch.no_grad():
a = torch.randn(2, 2, requires_grad=True)
print(a.requires_grad)
with torch.no_grad():
    b = a ** 2
    print(b.requires_grad)

True
False


# Linear Regression Gradient Descent with Autograd
eg -
f(x)=w∗x+b

here : `f(x) = 2 * x`

In [14]:
# initializing training data x and y
X = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8], dtype=torch.float32)
Y = torch.tensor([2, 4, 6, 8, 10, 12, 14, 16], dtype=torch.float32)

# initilizing weight w, here we want to track the gradient
w = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

# forward function for model output
def forward(x):
  return w*x

# loss function - MSE
def loss(y, y_pred):
  return ((y_pred - y)**2).mean()


X_test = 5.0

print(f'Prediction before training: f({X_test}) = {forward(X_test).item():.3f}')

Prediction before training: f(5.0) = 0.000


In [15]:
#training
learning_rate = 0.01
n_epochs = 100

for epoch in range(n_epochs):
    # predict = forward pass
    y_pred = forward(X)

    # loss
    l = loss(Y, y_pred)

    # calculate gradients = backward pass
    l.backward() # dl/dw

    # update weights
    # w.data = w.data - learning_rate * w.grad
    with torch.no_grad(): # since we do not want to track weight update calculation in gradiend calculation
      w -= learning_rate * w.grad

     # zero the gradients after updating
    w.grad.zero_() # we need to enpty the gradient for next epoch iteration

    if (epoch+1) % 10 == 0:
        print(f'epoch {epoch+1}: w = {w.item():.3f}, loss = {l.item():.3f}')

print(f'Prediction after training: f({X_test}) = {forward(X_test).item():.3f}')



epoch 10: w = 1.998, loss = 0.000
epoch 20: w = 2.000, loss = 0.000
epoch 30: w = 2.000, loss = 0.000
epoch 40: w = 2.000, loss = 0.000
epoch 50: w = 2.000, loss = 0.000
epoch 60: w = 2.000, loss = 0.000
epoch 70: w = 2.000, loss = 0.000
epoch 80: w = 2.000, loss = 0.000
epoch 90: w = 2.000, loss = 0.000
epoch 100: w = 2.000, loss = 0.000
Prediction after training: f(5.0) = 10.000
